Code Ran on Google Collab:

In [ ]:
"""
ICA Statistical Analysis — fills Table 1 and Table 2
=====================================================
Subsection: "Statistical Analysis and Interpretability of ICA Components"

Table 1: Comparative statistical diagnostics for ICA latent spaces
         with and without GA (12, 32, 53 components)

Table 2: Interpretation of the ICA components in the best ICA-GA model
         (12 components)

Builds directly on the original ICA_GA_pipeline.py logic.
Author: Adam Abdul Karim & Nour Harajli
"""

import time
import warnings
import numpy as np
import pandas as pd
import random
from itertools import combinations

from sklearn.decomposition import FastICA
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.stats import kruskal, spearmanr
from statsmodels.stats.multitest import multipletests
from deap import base, creator, tools, algorithms

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

# ─────────────────────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
print("Loading data...")
X_raw = pd.read_excel('../minmax.xlsx').values.astype(np.float32)
y_raw = pd.read_csv('../idC_with_header.csv')['Label'].values

# Wavenumbers: 900 features spanning 500-4000 cm^-1 linearly
wavenumbers = np.linspace(500, 4000, X_raw.shape[1])

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

# Ordered maturity index (0=least mature ... 13=most mature)
# Maps each class label to its maturity rank
unique_classes = sorted(np.unique(y_raw))
maturity_index = {cls: i for i, cls in enumerate(unique_classes)}
y_train_maturity = np.array([maturity_index[c] for c in y_train])

print(f"Data loaded: {X_raw.shape[0]} samples x {X_raw.shape[1]} features")
print(f"Wavenumber range: {wavenumbers[0]:.0f} - {wavenumbers[-1]:.0f} cm^-1\n")

# ─────────────────────────────────────────────────────────────────────────────
# 2. GA HELPER (SVM fitness — best performing from original results)
# ─────────────────────────────────────────────────────────────────────────────
def run_ga(X_tr, y_tr, fitness_clf, n_components, n_runs=10):
    """
    Runs GA n_runs times and tracks how often each component is selected.
    Returns: best selected components + selection frequency per component.
    """
    selection_counts = np.zeros(n_components)
    best_selected    = None
    best_fitness     = 0.0

    for run in range(n_runs):
        for name in ("FitnessMax", "Individual"):
            if name in creator.__dict__:
                delattr(creator, name)

        creator.create("FitnessMax", base.Fitness, weights=(1.0,))
        creator.create("Individual", list, fitness=creator.FitnessMax)

        tb = base.Toolbox()
        tb.register("attr_bool",  random.randint, 0, 1)
        tb.register("individual", tools.initRepeat, creator.Individual,
                    tb.attr_bool, n=n_components)
        tb.register("population", tools.initRepeat, list, tb.individual)

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=run)

        def fitness_fn(individual):
            selected = [i for i, v in enumerate(individual) if v == 1]
            if len(selected) < 2:
                return (0.0,)
            scores = cross_val_score(fitness_clf, X_tr[:, selected], y_tr,
                                     cv=skf, scoring='accuracy')
            return (scores.mean(),)

        tb.register("evaluate", fitness_fn)
        tb.register("mate",     tools.cxTwoPoint)
        tb.register("mutate",   tools.mutFlipBit, indpb=0.05)
        tb.register("select",   tools.selTournament, tournsize=3)

        pop = tb.population(n=100)
        hof = tools.HallOfFame(1)
        algorithms.eaSimple(pop, tb, cxpb=0.7, mutpb=0.2, ngen=50,
                            halloffame=hof, verbose=False)

        best     = hof[0]
        selected = [i for i, v in enumerate(best) if v == 1]
        fit_val  = hof[0].fitness.values[0]

        for idx in selected:
            selection_counts[idx] += 1

        if fit_val > best_fitness:
            best_fitness  = fit_val
            best_selected = selected

    selection_freq = (selection_counts / n_runs) * 100  # percentage
    return best_selected, selection_freq

# ─────────────────────────────────────────────────────────────────────────────
# 3. STATISTICAL HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def kruskal_wallis_per_component(scores, labels):
    """
    For each IC, test if scores differ significantly across 14 maturity classes.
    Returns H statistics and raw p-values.
    """
    n_components = scores.shape[1]
    H_vals, p_vals = [], []
    for ic in range(n_components):
        groups = [scores[labels == cls, ic] for cls in np.unique(labels)]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) < 2:
            H_vals.append(0.0); p_vals.append(1.0)
            continue
        H, p = kruskal(*groups)
        H_vals.append(H); p_vals.append(p)
    return np.array(H_vals), np.array(p_vals)

def effect_size_eta_squared(scores, labels):
    """
    Eta-squared effect size for each IC (analogous to R² for Kruskal-Wallis).
    eta² = (H - k + 1) / (n - k)  where k=number of groups, n=total samples
    """
    n = len(labels)
    k = len(np.unique(labels))
    H_vals, _ = kruskal_wallis_per_component(scores, labels)
    eta2 = (H_vals - k + 1) / (n - k)
    eta2 = np.clip(eta2, 0, 1)
    return eta2

def spearman_with_maturity(scores, maturity_indices):
    """
    Spearman correlation of each IC score with ordered maturity index.
    """
    rhos = []
    for ic in range(scores.shape[1]):
        rho, _ = spearmanr(scores[:, ic], maturity_indices)
        rhos.append(rho)
    return np.array(rhos)

def mean_inter_component_correlation(scores):
    """
    Average absolute pairwise Pearson correlation among all IC scores.
    Low value = components are independent (good for ICA).
    High value = redundancy.
    """
    n = scores.shape[1]
    if n < 2:
        return 0.0
    corr_matrix = np.corrcoef(scores.T)
    pairs = [(i, j) for i, j in combinations(range(n), 2)]
    mean_corr = np.mean([abs(corr_matrix[i, j]) for i, j in pairs])
    return mean_corr

def loading_concentration(mixing_matrix, top_n=10):
    """
    For each IC, what fraction of total |loading| is held by the top_n wavelengths?
    c_k = sum(top_n |loadings|) / sum(all |loadings|)
    Higher = more concentrated = more interpretable.
    """
    n_components = mixing_matrix.shape[1]
    concentrations = []
    for ic in range(n_components):
        loadings  = np.abs(mixing_matrix[:, ic])
        top_sum   = np.sum(np.sort(loadings)[::-1][:top_n])
        total_sum = np.sum(loadings)
        concentrations.append(top_sum / total_sum if total_sum > 0 else 0)
    return np.array(concentrations)

def bootstrap_stability(X_tr, y_tr, n_components, n_boots=20):
    """
    Refit ICA n_boots times on 80% subsamples.
    Match components across runs by maximum absolute correlation.
    Return mean cosine similarity of matched loading vectors.
    """
    reference_ica = FastICA(n_components=n_components, max_iter=1000,
                            random_state=42, whiten='unit-variance')
    reference_ica.fit(X_tr)
    ref_components = reference_ica.components_  # shape: (n_components, n_features)

    similarities = []
    n_samples = X_tr.shape[0]

    for boot in range(n_boots):
        rng     = np.random.RandomState(boot)
        indices = rng.choice(n_samples, size=int(0.8 * n_samples), replace=False)
        X_boot  = X_tr[indices]

        try:
            boot_ica = FastICA(n_components=n_components, max_iter=1000,
                               random_state=boot, whiten='unit-variance')
            boot_ica.fit(X_boot)
            boot_components = boot_ica.components_

            # Match each reference component to best-matching boot component
            for ref_comp in ref_components:
                corrs = [abs(np.corrcoef(ref_comp, bc)[0, 1])
                         for bc in boot_components]
                similarities.append(max(corrs))
        except Exception:
            continue

    return np.mean(similarities) if similarities else 0.0

def cv_accuracy(X_latent, y, n_folds=5):
    """Cross-validated SVM accuracy on the latent space."""
    clf = SVC(random_state=42)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    scores = cross_val_score(clf, X_latent, y, cv=skf, scoring='accuracy')
    return scores.mean(), scores.std()

# ─────────────────────────────────────────────────────────────────────────────
# 4. TABLE 1 — run for n_components = 12, 32, 53
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("COMPUTING TABLE 1 — Comparative diagnostics across ICA settings")
print("=" * 70)

component_settings = [12, 32, 53]
table1_rows = []

# Known accuracies from your original results
known_acc = {
    (12, False): (88.76, 0.89),
    (12, True):  (91.01, 0.91),
    (32, False): (79.78, None),
    (32, True):  (None,  None),
    (53, False): (76.40, None),
    (53, True):  (None,  None),
}

for n_comp in component_settings:
    print(f"\n  Fitting ICA with {n_comp} components...")
    t0 = time.time()

    ica = FastICA(n_components=n_comp, max_iter=1000,
                  random_state=42, whiten='unit-variance')
    X_train_ica = ica.fit_transform(X_train_raw)
    X_test_ica  = ica.transform(X_test_raw)
    mixing      = ica.mixing_  # shape: (900, n_comp)

    print(f"    ICA fit done in {time.time()-t0:.1f}s")

    # --- ICA alone (no GA) ---
    cv_mean, cv_std = cv_accuracy(X_train_ica, y_train)

    H_vals, p_vals = kruskal_wallis_per_component(X_train_ica, y_train)
    _, q_vals, _, _ = multipletests(p_vals, method='fdr_bh')
    n_sig     = int(np.sum(q_vals < 0.05))
    eta2      = effect_size_eta_squared(X_train_ica, y_train)
    mean_eta2 = float(np.mean(eta2))
    mean_corr = mean_inter_component_correlation(X_train_ica)
    conc      = loading_concentration(mixing, top_n=10)
    mean_conc = float(np.mean(conc))

    print(f"    Computing bootstrap stability (no GA)...")
    stab = bootstrap_stability(X_train_raw, y_train, n_comp, n_boots=20)

    acc_no_ga, f1_no_ga = known_acc.get((n_comp, False), (None, None))

    table1_rows.append({
        "Pipeline"      : "ICA",
        "ICs"           : n_comp,
        "GA"            : "No",
        "Acc. (%)"      : f"{acc_no_ga:.2f}" if acc_no_ga else "—",
        "Macro-F1"      : f"{f1_no_ga:.2f}" if f1_no_ga else "—",
        "CV Acc mean±SD": f"{cv_mean*100:.2f}±{cv_std*100:.2f}",
        "Sig. ICs (q<0.05)": n_sig,
        "Mean effect size": f"{mean_eta2:.3f}",
        "Mean |ρ| inter-IC": f"{mean_corr:.3f}",
        "Loading conc."  : f"{mean_conc:.3f}",
        "Bootstrap stab.": f"{stab:.3f}",
        "ICs kept by GA" : n_comp,
    })

    # --- ICA + GA ---
    print(f"    Running GA (SVM fitness, 10 runs) for {n_comp} components...")
    t0 = time.time()
    fit_clf = SVC(random_state=42)
    best_selected, sel_freq = run_ga(X_train_ica, y_train, fit_clf, n_comp, n_runs=10)

    X_tr_sel = X_train_ica[:, best_selected]
    cv_mean_ga, cv_std_ga = cv_accuracy(X_tr_sel, y_train)

    # Recompute stats on selected components only
    H_sel, p_sel  = kruskal_wallis_per_component(X_train_ica[:, best_selected], y_train)
    _, q_sel, _, _ = multipletests(p_sel, method='fdr_bh')
    n_sig_ga   = int(np.sum(q_sel < 0.05))
    eta2_sel   = effect_size_eta_squared(X_train_ica[:, best_selected], y_train)
    mean_eta2_ga = float(np.mean(eta2_sel))
    mean_corr_ga = mean_inter_component_correlation(X_train_ica[:, best_selected])
    conc_sel   = loading_concentration(mixing[:, best_selected], top_n=10)
    mean_conc_ga = float(np.mean(conc_sel))

    acc_ga, f1_ga = known_acc.get((n_comp, True), (None, None))

    print(f"    GA done in {time.time()-t0:.1f}s — selected {len(best_selected)}/{n_comp} components")

    table1_rows.append({
        "Pipeline"         : "ICA-GA",
        "ICs"              : n_comp,
        "GA"               : "Yes",
        "Acc. (%)"         : f"{acc_ga:.2f}" if acc_ga else "—",
        "Macro-F1"         : f"{f1_ga:.2f}" if f1_ga else "—",
        "CV Acc mean±SD"   : f"{cv_mean_ga*100:.2f}±{cv_std_ga*100:.2f}",
        "Sig. ICs (q<0.05)": n_sig_ga,
        "Mean effect size" : f"{mean_eta2_ga:.3f}",
        "Mean |ρ| inter-IC": f"{mean_corr_ga:.3f}",
        "Loading conc."    : f"{mean_conc_ga:.3f}",
        "Bootstrap stab."  : f"{stab:.3f}",  # stability is property of ICA not GA
        "ICs kept by GA"   : len(best_selected),
    })

# Print Table 1
print("\n\n" + "=" * 70)
print("TABLE 1 — Comparative statistical diagnostics for ICA latent spaces")
print("          with and without GA")
print("=" * 70)
df_table1 = pd.DataFrame(table1_rows)
print(df_table1.to_string(index=False))
df_table1.to_csv("ICA_Table1_diagnostics.csv", index=False)
print("\nSaved to: ICA_Table1_diagnostics.csv")

# ─────────────────────────────────────────────────────────────────────────────
# 5. TABLE 2 — deep dive into best model: ICA-GA, 12 components
# ─────────────────────────────────────────────────────────────────────────────
print("\n\n" + "=" * 70)
print("COMPUTING TABLE 2 — Per-component interpretation (ICA-GA, n=12)")
print("=" * 70)

# Refit ICA with 12 components
ica12 = FastICA(n_components=12, max_iter=1000,
                random_state=42, whiten='unit-variance')
X_train_ica12 = ica12.fit_transform(X_train_raw)
X_test_ica12  = ica12.transform(X_test_raw)
mixing12      = ica12.mixing_   # shape: (900, 12)

# Run GA 10 times to get selection frequencies
print("Running GA 10 times to get selection frequencies...")
fit_clf = SVC(random_state=42)
best_sel12, sel_freq12 = run_ga(X_train_ica12, y_train, fit_clf,
                                 n_components=12, n_runs=10)

# Per-component statistics
H_vals12, p_vals12 = kruskal_wallis_per_component(X_train_ica12, y_train)
_, q_vals12, _, _  = multipletests(p_vals12, method='fdr_bh')
eta2_12            = effect_size_eta_squared(X_train_ica12, y_train)
rhos12             = spearman_with_maturity(X_train_ica12, y_train_maturity)
conc12             = loading_concentration(mixing12, top_n=10)

TOP_N_WAVENUMBERS = 5  # report top 5 wavenumbers per component

table2_rows = []
for ic in range(12):
    loadings    = mixing12[:, ic]
    abs_load    = np.abs(loadings)
    top_indices = np.argsort(abs_load)[::-1][:TOP_N_WAVENUMBERS]
    top_wns     = [f"{wavenumbers[i]:.0f}" for i in top_indices]
    top_wns_str = ", ".join(top_wns) + " cm⁻¹"

    # Tentative spectral interpretation based on known MIR bands
    # Reference: common MIR assignments for fruit/strawberry
    dominant_wn = wavenumbers[top_indices[0]]
    if 900 <= dominant_wn <= 1200:
        interpretation = "C-O / C-C stretch (sugars, polysaccharides)"
    elif 1200 <= dominant_wn <= 1400:
        interpretation = "C-H bending / carboxylate stretch"
    elif 1400 <= dominant_wn <= 1600:
        interpretation = "C=C / COO⁻ stretch (organic acids)"
    elif 1600 <= dominant_wn <= 1700:
        interpretation = "Amide I / C=O stretch (proteins)"
    elif 1700 <= dominant_wn <= 1800:
        interpretation = "C=O ester stretch (lipids, esters)"
    elif 2800 <= dominant_wn <= 3000:
        interpretation = "C-H stretch (lipids, waxes)"
    elif 3000 <= dominant_wn <= 3600:
        interpretation = "O-H stretch (water, alcohols)"
    else:
        interpretation = "Mixed / overlapping bands"

    sig_marker = "*" if q_vals12[ic] < 0.05 else ""

    table2_rows.append({
        "IC"                        : f"IC{ic+1}",
        "GA sel. freq. (%)"         : f"{sel_freq12[ic]:.0f}%",
        "Kruskal-Wallis H"          : f"{H_vals12[ic]:.2f}",
        "q-value"                   : f"{q_vals12[ic]:.4f}{sig_marker}",
        "Effect size (η²)"          : f"{eta2_12[ic]:.3f}",
        "Spearman ρ w/ maturity"    : f"{rhos12[ic]:.3f}",
        f"Top {TOP_N_WAVENUMBERS} MIR markers": top_wns_str,
        "Loading conc."             : f"{conc12[ic]:.3f}",
        "Tentative interpretation"  : interpretation,
    })

print("\n\nTABLE 2 — Interpretation of ICA components (ICA-GA, 12 components)")
print("* = significant after Benjamini-Hochberg correction (q < 0.05)")
print("=" * 70)
df_table2 = pd.DataFrame(table2_rows)
print(df_table2.to_string(index=False))
df_table2.to_csv("ICA_Table2_component_interpretation.csv", index=False)
print("\nSaved to: ICA_Table2_component_interpretation.csv")

# ─────────────────────────────────────────────────────────────────────────────
# 6. AUTO-GENERATE THE INTERPRETATION PARAGRAPH
# ─────────────────────────────────────────────────────────────────────────────
print("\n\n" + "=" * 70)
print("AUTO-GENERATED INTERPRETATION PARAGRAPH")
print("(copy and edit for your paper)")
print("=" * 70)

# Find most discriminative IC (highest η²)
best_ic_idx  = int(np.argmax(eta2_12))
best_ic_name = f"IC{best_ic_idx + 1}"
best_rho     = rhos12[best_ic_idx]
best_wns     = table2_rows[best_ic_idx][f"Top {TOP_N_WAVENUMBERS} MIR markers"]
best_interp  = table2_rows[best_ic_idx]["Tentative interpretation"]
best_ga_freq = sel_freq12[best_ic_idx]

# Most frequently GA-selected IC
most_selected_idx  = int(np.argmax(sel_freq12))
most_selected_name = f"IC{most_selected_idx + 1}"
most_selected_freq = sel_freq12[most_selected_idx]

n_sig_total = int(np.sum(q_vals12 < 0.05))

paragraph = f"""
The superior performance of the 12-component ICA-GA model reflects a balance between
information retention and latent compactness. Of the 12 independent components, {n_sig_total}
showed statistically significant class separation across the 14 maturity levels after
Benjamini-Hochberg correction (q < 0.05). {best_ic_name} exhibited the strongest
discriminative power (η² = {eta2_12[best_ic_idx]:.3f}) and a Spearman correlation of
ρ = {best_rho:.3f} with the ordered maturity index, suggesting a consistent association
with maturity progression. {best_ic_name} was primarily driven by spectral markers around
{best_wns}, consistent with {best_interp}. The GA stage further refined the latent
representation by selectively retaining the most informative components; {most_selected_name}
was selected in {most_selected_freq:.0f}% of GA runs, confirming its stable discriminative
contribution. Relative to the 32- and 53-component settings, the 12-component solution
avoids the inclusion of weaker, noisier, or redundant latent factors, thereby improving
generalization and strengthening interpretability of the MIR spectral signatures associated
with strawberry maturity.
"""
print(paragraph)

with open("ICA_interpretation_paragraph.txt", "w") as f:
    f.write(paragraph)
print("Saved to: ICA_interpretation_paragraph.txt")